# Traffic signals — static get / set demo

Minimal **standalone** notebook:

1. Load a level with `signals.json` (**`west_coast_usa`** or **`east_coast_usa`**).
2. Spawn a vehicle and **teleport it in front of one traffic light** (`get_traffic_light` → `vehicle.teleport`).
3. **GET** current lamp state, **SET** a strict override, **GET** again, then **clear** override.
4. **GET / SET** controller **Duration** values from Traffic Signals Editor data (then restore).

The vehicle stays put (no traffic AI). Signal phases can still change on their own until you use `set_instance_strict_state`.

| API | Role |
|-----|------|
| `get_traffic_light(name)` | Pose (`pos`, `rot`) + current `state` / `action` |
| `get_instance_state(name)` | Logical state only |
| `set_instance_strict_state(name, idx)` | Force controller state; returns `before` / `after` |
| `set_instance_strict_state(name, None)` | Restore automatic control |
| `get_editor_signals(name)` | Controller **Duration** + sequence **phases** (Traffic Signals Editor / `signals.json`) |
| `set_controller_state_duration(ctrl, idx, sec)` | Same **Duration** field as editor (runtime only; Save in WE for disk) |

## 1. Connect + configuration

Set **`LEVEL`** and **`TARGET_SIGNAL`** (must match that level’s instance names from Traffic Manager / `signals.json`).

- West Coast: `trafficLight 1` (with space)
- East Coast: `trafficLight1` (no space)

In [29]:
from pprint import pprint
from time import sleep

from beamngpy import BeamNGpy, Scenario, Vehicle
from beamngpy.logging import BNGError

HOST, PORT = "localhost", 25252
LAUNCH = True  # True only when BeamNG is not running yet

# LEVEL = "west_coast_usa"
LEVEL = "east_coast_usa"

# TARGET_SIGNAL = "trafficLight 1"   # west_coast_usa
TARGET_SIGNAL = "trafficLight1"    # east_coast_usa

DISTANCE_BEFORE_LIGHT_M = 15.0

bng = BeamNGpy(HOST, PORT, quit_on_close=False)
bng.open(launch=LAUNCH)
print("connected")

connected


## 2. Load scenario, spawn vehicle, park in front of the light

No traffic AI — static demo only.

**Re-run tip:** each run uses a **unique scenario name** so GE does not reject a duplicate prefab.
If §1 reconnects while the level is already loaded, expect a full level reload (~1–2 min on East Coast).

In [30]:
spawned_traffic_here = False
demo_vehicle = None


scenario_name = "traffic signals demo"
scenario = Scenario(LEVEL, scenario_name)
demo_vehicle = Vehicle("static_demo", model="etk800", license="TS", color="Blue")
scenario.add_vehicle(
    demo_vehicle,
    pos=(0, 0, 0),
    rot_quat=(0, 0, 0, 1),
    safe_spawn=True,
)
scenario.make(bng)
bng.settings.set_deterministic(60)
print(f"loading scenario {scenario_name!r} on {LEVEL!r} …", flush=True)
bng.scenario.load(scenario, precompile_shaders=False)
bng.ui.hide_hud()
bng.scenario.start()
try:
    bng.control.resume()
except Exception:
    pass

sleep(10)

demo_vehicle.ai.set_mode("disabled")

traffic_light_instance = bng.traffic_signals.get_traffic_light(
    TARGET_SIGNAL, distance_m=DISTANCE_BEFORE_LIGHT_M
)
demo_vehicle.teleport(
    traffic_light_instance["pos"],
    traffic_light_instance["rot"],
    reset=True,
    safe_spawn=True,
)
print(f"Vehicle parked before {TARGET_SIGNAL!r} on {LEVEL!r}")
pprint(traffic_light_instance)

loading scenario 'traffic signals demo' on 'east_coast_usa' …
Vehicle parked before 'trafficLight1' on 'east_coast_usa'
{'action': 'stop',
 'name': 'trafficLight1',
 'pos': (728.645460715234, -61.77420716489588, 51.83232599430656),
 'rot': (0.0, 0.0, -0.9999992036382469, 0.001262031248418117),
 'signal_dir': (-0.0025237143489333868,
                -0.9998596796736086,
                0.01656055037956289),
 'signal_pos': (728.607605, -76.77210236, 52.08073425),
 'state': 'redTrafficLight'}


## 3. GET — read traffic light state

`get_traffic_light` includes pose + live state; `get_instance_state` is state-only.

In [31]:
state_now = bng.traffic_signals.get_instance_state(TARGET_SIGNAL)
print("GET get_instance_state:")
pprint(state_now)

traffic_light_instance = bng.traffic_signals.get_traffic_light(
    TARGET_SIGNAL, distance_m=DISTANCE_BEFORE_LIGHT_M
)
print("GET get_traffic_light (state + pose):")
print(
    f"  state={traffic_light_instance['state']!r} "
    f"action={traffic_light_instance.get('action')!r}"
)

GET get_instance_state:
{'action': 'stop',
 'controller_id': 4.0,
 'dir': [-0.0025237143490000006, -0.9998596797, 0.016560550380000007],
 'name': 'trafficLight1',
 'pos': [728.607605, -76.77210236, 52.08073425],
 'sequence_id': 7.0,
 'state': 'redTrafficLight'}
GET get_traffic_light (state + pose):
  state='redTrafficLight' action='stop'


## 4. SET — override state (returns **before** / **after**)

Change `OVERRIDE_STATE_IDX` to another phase index, or re-run after the sequence timer changes the light.

In [32]:
OVERRIDE_STATE_IDX = 2  # 1-based; try 1=green, 2=yellow, 3=red on lightsBasic

override_report = bng.traffic_signals.set_instance_strict_state(
    TARGET_SIGNAL, OVERRIDE_STATE_IDX
)
print("SET set_instance_strict_state:")
print("  before:", override_report["before"])
print("  after: ", override_report["after"])

SET set_instance_strict_state:
  before: {'state': 'redTrafficLight', 'name': 'trafficLight1', 'action': 'stop', 'controller_id': 4.0, 'pos': [728.607605, -76.77210236, 52.08073425], 'sequence_id': 7.0, 'dir': [-0.0025237143490000006, -0.9998596797, 0.016560550380000007]}
  after:  {'state': 'yellowTrafficLight', 'name': 'trafficLight1', 'action': 'alert', 'controller_id': 4.0, 'pos': [728.607605, -76.77210236, 52.08073425], 'sequence_id': 7.0, 'dir': [-0.0025237143490000006, -0.9998596797, 0.016560550380000007]}


## 5. GET again — confirm override, then clear (SET `None`)

In [33]:
after_override = bng.traffic_signals.get_instance_state(TARGET_SIGNAL)
print("GET after override:")
pprint(after_override)

clear_report = bng.traffic_signals.set_instance_strict_state(TARGET_SIGNAL, None)
print("SET clear (state_index=None):")
print("  before:", clear_report["before"])
print("  after: ", clear_report["after"])

GET after override:
{'action': 'alert',
 'controller_id': 4.0,
 'dir': [-0.0025237143490000006, -0.9998596797, 0.016560550380000007],
 'name': 'trafficLight1',
 'pos': [728.607605, -76.77210236, 52.08073425],
 'sequence_id': 7.0,
 'state': 'yellowTrafficLight'}
SET clear (state_index=None):
  before: {'state': 'yellowTrafficLight', 'name': 'trafficLight1', 'action': 'alert', 'controller_id': 4.0, 'pos': [728.607605, -76.77210236, 52.08073425], 'sequence_id': 7.0, 'dir': [-0.0025237143490000006, -0.9998596797, 0.016560550380000007]}
  after:  {'state': 'redTrafficLight', 'name': 'trafficLight1', 'action': 'stop', 'controller_id': 4.0, 'pos': [728.607605, -76.77210236, 52.08073425], 'sequence_id': 7.0, 'dir': [-0.0025237143490000006, -0.9998596797, 0.016560550380000007]}


## 6. GET / SET — durations (Traffic Signals Editor data)

Timings come from the level **`signals.json`** loaded by GE — the same objects the **Traffic Signals Editor** edits (controller **States** → **Duration** column; sequence **Phases** table).

- **GET:** each **sequence phase**, then each **controller** in that phase with every **State → Duration** row (editor / `signals.json`).
- **SET:** `set_controller_state_duration` on those controllers; phase **totalDuration** is recalculated in GE. Does **not** write `signals.json`; use editor **Save** to persist.

In [34]:
def _dur_str(dur):
    if dur is None:
        return "?"
    if float(dur) < 0:
        return "infinite"
    return f"{float(dur):.3f} s"


def _controllers_by_id(level_editor):
    return {c["id"]: c for c in level_editor["controllers"]}


editor = bng.traffic_signals.get_editor_signals(TARGET_SIGNAL)
level = bng.traffic_signals.get_editor_signals()
ctrl_by_id = _controllers_by_id(level)
ctrl = editor["controller"]
seq = editor.get("sequence")

print(f"GET get_editor_signals({TARGET_SIGNAL!r})")
print(f"  lamp controllerId={editor['instance']['controllerId']} sequenceId={editor['instance']['sequenceId']}")
print(f"  this lamp's controller {ctrl['name']!r} ({ctrl.get('type')!r}):")
for i, st in enumerate(ctrl["states"], start=1):
    print(f"    state row {i}: {st.get('state')!r}  duration={_dur_str(st.get('duration'))}")

if seq:
    print(f"\n  sequence {seq['name']!r} (editor Phases table)")
    for phase_i, ph in enumerate(seq.get("phases") or [], start=1):
        total = ph.get("totalDuration")
        total_s = "infinite" if total is not None and float(total) >= 1e30 else f"{float(total):.3f} s"
        print(f"\n  --- phase {phase_i} ---  totalDuration={total_s}  autoReset={ph.get('autoReset')}")
        for cid in ph.get("controllerIds") or []:
            c = ctrl_by_id.get(cid)
            if not c:
                print(f"    controller id {cid}: (not found)")
                continue
            print(f"    controller [{cid}] {c['name']!r}:")
            for row_i, st in enumerate(c.get("states") or [], start=1):
                print(f"      state row {row_i}: {st.get('state')!r}  duration={_dur_str(st.get('duration'))}")

print(f"\n  global timer: {editor.get('timer')} s")

GET get_editor_signals('trafficLight1')
  lamp controllerId=4.0 sequenceId=7.0
  this lamp's controller 'green15_EW' ('lightsBasic'):
    state row 1: 'greenTrafficLight'  duration=15.000 s
    state row 2: 'yellowTrafficLight'  duration=4.000 s
    state row 3: 'redTrafficLight'  duration=1.250 s

  sequence 'trafficLights_10_15' (editor Phases table)

  --- phase 1 ---  totalDuration=15.250 s  autoReset=None
    controller [1.0] 'green10_NS':
      state row 1: 'greenTrafficLight'  duration=10.000 s
      state row 2: 'yellowTrafficLight'  duration=4.000 s
      state row 3: 'redTrafficLight'  duration=1.250 s

  --- phase 2 ---  totalDuration=20.250 s  autoReset=None
    controller [4.0] 'green15_EW':
      state row 1: 'greenTrafficLight'  duration=15.000 s
      state row 2: 'yellowTrafficLight'  duration=4.000 s
      state row 3: 'redTrafficLight'  duration=1.250 s

  global timer: 8.350000124424696 s


In [35]:
# New durations: controller_name -> {1-based state row: seconds}
# Controllers listed under each sequence phase above. Empty dict = demo bumps green (row 1) by +1.0 s each.
NEW_DURATIONS = {
     "green10_NS": {1: 12.0, 2: 4.0, 3: 1.5},
     "green10_EW": {1: 12.0, 2: 4.0, 3: 1.5},
}
BUMP_GREEN_BY_SEC = 1.0  # used when NEW_DURATIONS is empty

editor_before = bng.traffic_signals.get_editor_signals(TARGET_SIGNAL)
level = bng.traffic_signals.get_editor_signals()
ctrl_by_id = _controllers_by_id(level)
seq = editor_before["sequence"]

# Collect controllers used in this sequence (once per name)
controllers_in_seq = {}
for ph in seq.get("phases") or []:
    for cid in ph.get("controllerIds") or []:
        c = ctrl_by_id.get(cid)
        if c:
            controllers_in_seq[c["name"]] = c

to_apply = []
for cname, ctrl in sorted(controllers_in_seq.items()):
    changes = NEW_DURATIONS.get(cname)
    if not changes:
        row = 1
        old = float(ctrl["states"][row - 1]["duration"])
        changes = {row: old + BUMP_GREEN_BY_SEC}
    for row, new_dur in sorted(changes.items()):
        old = float(ctrl["states"][row - 1]["duration"])
        st_name = ctrl["states"][row - 1].get("state")
        to_apply.append((cname, row, st_name, old, float(new_dur)))

print("SET new durations (controller state rows from editor):")
restores = []
for cname, row, st_name, old, new_dur in to_apply:
    print(f"  {cname!r} row {row} ({st_name!r}): {old:.3f} -> {new_dur:.3f} s")
    bng.traffic_signals.set_controller_state_duration(
        cname, row, new_dur, instance_name=TARGET_SIGNAL
    )
    restores.append((cname, row, old))

print("\nGET phases after SET:")
editor_after = bng.traffic_signals.get_editor_signals(TARGET_SIGNAL)
ctrl_by_id_after = _controllers_by_id(bng.traffic_signals.get_editor_signals())
seq_after = editor_after["sequence"]
for phase_i, ph in enumerate(seq_after.get("phases") or [], start=1):
    total = ph.get("totalDuration")
    total_s = "infinite" if total is not None and float(total) >= 1e30 else f"{float(total):.3f} s"
    print(f"  phase {phase_i}: totalDuration={total_s}")
    for cid in ph.get("controllerIds") or []:
        c = ctrl_by_id_after.get(cid)
        if c:
            print(f"    {c['name']!r}: " + ", ".join(
                f"row{r}={_dur_str(c['states'][r-1].get('duration'))}"
                for r in range(1, len(c.get("states") or []) + 1)
            ))

print("\nSET restore original durations:")
for cname, row, old in restores:
    bng.traffic_signals.set_controller_state_duration(cname, row, old)
    print(f"  {cname!r} row {row}: restored {old:.3f} s")

SET new durations (controller state rows from editor):
  'green10_NS' row 1 ('greenTrafficLight'): 10.000 -> 12.000 s
  'green10_NS' row 2 ('yellowTrafficLight'): 4.000 -> 4.000 s
  'green10_NS' row 3 ('redTrafficLight'): 1.250 -> 1.500 s
  'green15_EW' row 1 ('greenTrafficLight'): 15.000 -> 16.000 s

GET phases after SET:
  phase 1: totalDuration=17.500 s
    'green10_NS': row1=12.000 s, row2=4.000 s, row3=1.500 s
  phase 2: totalDuration=21.250 s
    'green15_EW': row1=16.000 s, row2=4.000 s, row3=1.250 s

SET restore original durations:
  'green10_NS' row 1: restored 10.000 s
  'green10_NS' row 2: restored 4.000 s
  'green10_NS' row 3: restored 1.250 s
  'green15_EW' row 1: restored 15.000 s


## 7. Optional — list instances on this level (pick another `TARGET_SIGNAL`)

In [36]:
instances = bng.traffic_signals.list_instances()
lights = [i for i in instances if "trafficLight" in i.get("name", "")]
print(f"traffic lights on {LEVEL!r}: {len(lights)} (total instances {len(instances)})")
pprint(lights[:8])

traffic lights on 'east_coast_usa': 9 (total instances 34)
[{'action': 'stop',
  'controller_id': 4.0,
  'name': 'trafficLight1',
  'pos': [728.607605, -76.77210236, 52.08073425],
  'sequence_id': 7.0,
  'state': 'redTrafficLight'},
 {'action': 'stop',
  'controller_id': 4.0,
  'name': 'trafficLight2',
  'pos': [734.586792, -105.9545212, 52.85263062],
  'sequence_id': 7.0,
  'state': 'redTrafficLight'},
 {'action': 'go',
  'controller_id': 1.0,
  'name': 'trafficLight3',
  'pos': [716.1392212, -85.89792633, 52.51080704],
  'sequence_id': 7.0,
  'state': 'greenTrafficLight'},
 {'action': 'stop',
  'controller_id': 4.0,
  'name': 'trafficLight4',
  'pos': [728.3527832, -149.2577667, 53.24858093],
  'sequence_id': 7.0,
  'state': 'redTrafficLight'},
 {'action': 'stop',
  'controller_id': 4.0,
  'name': 'trafficLight5',
  'pos': [734.5186157, -184.7062225, 53.35800934],
  'sequence_id': 7.0,
  'state': 'redTrafficLight'},
 {'action': 'go',
  'controller_id': 1.0,
  'name': 'trafficLight6',

## 8. Disconnect

In [37]:
try:
    demo_vehicle.ai.set_mode("disabled")
except Exception:
    pass
try:
    bng.ui.show_hud()
except Exception:
    pass
bng.disconnect()
print("disconnected")

disconnected
